# QAOA
VQEアルゴリズムとほぼ同じですが、組合せ最適化問題向けにQAOA専用のAnsatzを使用します。

## 今回学ぶこと
1. QAOAの仕組み
2. 簡単な例を使ったQAOAの実装

## 量子断熱計算
QAOAは、量子断熱計算と呼ばれる時間発展を用いた仕組みを利用します。

現在時刻を $t$、全体のスケジュールを $T$ とします。今回は2つのハミルトニアンを使います。最初に用意する初期ハミルトニアンを $H_{start}$、解きたい問題を含むハミルトニアンを $H_{final}$ とします。このとき、以下のように初期ハミルトニアンから解きたい問題のハミルトニアンへ徐々に切り替えていきます。

$$
H_{temp} = (1-\frac{t}{T})H_{start} + \frac{t}{T}H_{final}
$$

$T\rightarrow\infty$ であれば、基底状態は次の固有状態になります。

$$
H_{temp}\mid \psi \rangle = E_{0temp}\mid \psi \rangle
$$

2つのハミルトニアンを置き換えていく過程は、時間発展演算子を用いて作られます。

$$
\mid \psi_{t+1} \rangle = e^{-iHt}  \mid \psi_t \rangle
$$

## 量子回路
量子回路は次のもので構成されます。

1. 初期状態の準備
2. QAOA Ansatz

```
|0> ---[初期状態の準備]---[QAOA Ansatz]---[測定]
```

一番左が初期状態の準備、次がQAOA Ansatzの第1ステップ、その次が第2ステップです。最後に測定を行い、解を求めます。

## 2つのハミルトニアン
今回は、初期ハミルトニアン(ミキサーハミルトニアン)と、解きたい問題のためのハミルトニアン(コストハミルトニアン)を使用します。ミキサーハミルトニアンは問題の制約によって異なるものが選ばれます。コストハミルトニアンは、問題において最小化したい値を定式化したものです。

## ミキサーハミルトニアンと初期状態の準備
ミキサーハミルトニアンは、問題をどのように探索するかを決めるものです。量子アニーリングでは、量子ビットの値を反転させながら答えを探索するミキサーハミルトニアンとしてXが使われます。

ミキサーハミルトニアンが問題の制約によって決まると、初期状態も決まります。初期状態の準備では、ミキサーハミルトニアンの固有状態を設定します。上記のXをミキサーハミルトニアンとするXミキサーを使いたい場合は、デフォルトで $\mid + \rangle$ のような状態に設定された固有状態 $\mid \psi \rangle$ を選びます。

## コストハミルトニアン
コストハミルトニアンは問題によって定式化が変わります。社会的な問題をもとに、組合せ最適化問題と呼ばれる種類の問題は、物理学のイジングモデルと呼ばれるモデルに落とし込まれます。コストハミルトニアンは、Z演算子の+1と-1という期待値を用いて、ZまたはZZ演算子で問題の条件をすべて表現します。

```
hamiltonian = -Z(0) - Z(0)*Z(1)
```

単一のZの前に「バイアス」を、複数のZの前に「重み」を設定します。

## QAOA Ansatzとハミルトニアンの時間発展
QAOA Ansatzでは、問題を設定するコストハミルトニアンと探索方法を決めるミキサーハミルトニアンが、時間発展演算子と呼ばれる形で格納されます。

$X$ ゲートの時間発展は $RX(\theta)$ ゲートのような角度を含むゲートを利用します。2量子ビットの場合も同様に、$RXX(\theta)$ のような角度を含む $XX$ ゲートの時間発展が使われます。

## ステップ数
量子断熱計算は通常アナログのプロセスであり、時間発展演算子を用いて離散化されます。QAOAでもステップ数を設定することで精度を高めることができます。ステップ数はQAOA Ansatzの繰り返し回数に対応します。

※ 新しいBlueqatSDKでは、このステップ数がAnsatzの変分パラメータ数(勾配降下法で最適化されるレイヤー数)に直接対応するようになりました。旧バージョンのように大きなステップ数(数十〜数百)を指定すると回路が非常に深くなり最適化に時間がかかるため、このチュートリアルでは小さめのステップ数を使用しています。

## Xミキサー
典型的なミキサーはXミキサーです。

$$
X\mid 0 \rangle = 
\begin{bmatrix}
0&1\\
1&0
\end{bmatrix}
\begin{bmatrix}
1\\
0
\end{bmatrix}
=
\begin{bmatrix}
0\\
1
\end{bmatrix}
$$

これは単一の量子ビットに作用して、その量子ビットの値を反転させるために使われます。初期状態も、$\mid + \rangle$ 状態を作るために単一量子ビットにHゲートを適用します。

## XYミキサー
XYミキサーは2つの状態を入れ替えます。(XX+YY)/2 は

$$
X_0X_1 + Y_0Y_1 = 
\begin{bmatrix}
0&1\\
1&0
\end{bmatrix}
\otimes
\begin{bmatrix}
0&1\\
1&0
\end{bmatrix}
+
\begin{bmatrix}
0&-i\\
i&0
\end{bmatrix}
\otimes
\begin{bmatrix}
0&-i\\
i&0
\end{bmatrix}
=
\begin{bmatrix}
0&0&0&1\\
0&0&1&0\\
0&1&0&0\\
1&0&0&0
\end{bmatrix}
+
\begin{bmatrix}
0&0&0&-1\\
0&0&1&0\\
0&1&0&0\\
-1&0&0&0
\end{bmatrix}
$$

となるので、

$$
(X_0X_1 + Y_0Y_1)/2 
=
\begin{bmatrix}
0&0&0&0\\
0&0&1&0\\
0&1&0&0\\
0&0&0&0
\end{bmatrix}
$$

ここでは、01状態と10状態を入れ替えながら答えを探索します。初期状態には01と10のエンタングル状態が固有状態として選ばれます。また、

$$
(X_0X_1 - Y_0Y_1)/2 = 
\begin{bmatrix}
0&0&0&1\\
0&0&0&0\\
0&0&0&0\\
1&0&0&0
\end{bmatrix}
$$

これは00状態と11状態を入れ替えます。初期状態には00と11のエンタングル状態が選ばれます。

## 例: 量子アニーリング
QaoaAnsatzは初期状態では量子アニーリングに設定されており、初期状態の固有状態を|+>とするXミキサーを使っています。初期状態|+>は、すべての量子ビットにアダマールゲートHを適用することで実現されます。

```
|0> --H--  --RZZ--RZ--RX- -RZZ--RZ--RX--  --[測定]
|0> --H--  --*--------RX- -*--------RX--  --[測定]
```

In [1]:
import numpy as np
from blueqat import Circuit
from blueqat.utils import Vqe, QaoaAnsatz, X, Y, Z, I
from blueqat.utils import qubo_bit as q

#mixer
Xmixer = X[0]+X[1]

#initial state
Xinit = Circuit(2).h[0,1]

h = -Z(0) -Z(0)*Z(1)
step = 2

ansatz = QaoaAnsatz(h, step, Xinit, Xmixer)
result = Vqe(ansatz).run()
result.circuit.run(shots=10)

/Users/yuichirominato/blueqatSDK/.claude/worktrees/determined-mahavira-bf713e/blueqat/utils.py:399: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:823.)
  total_matrix = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.int64, device=device), torch.empty(0, dtype=torch.complex128, device=device), (dim, dim))


Counter({'11': 4, '00': 3, '10': 3})

初期状態とミキサーを指定しない場合は、量子アニーリングが採用されます。

In [2]:
ansatz = QaoaAnsatz(h, step)
result = Vqe(ansatz).run()
result.circuit.run(shots=10)

Counter({'00': 10})

## 例: XYミキサー
次に、同じ問題に対してXYミキサーを使ってみましょう。初期状態としてエンタングル状態を指定します。ミキサーは (XX+YY)/2 のハミルトニアンとして与えます。

In [3]:
#mixer
XYmixer = 0.5*X[0]*X[1] + 0.5*Y[0]*Y[1]

#initial state
XYinit = Circuit().h[0].cx[0,1].x[0]

ansatz = QaoaAnsatz(h, step, XYinit, XYmixer)
result = Vqe(ansatz).run()
b = result.circuit.run(shots=10)

print(b)

Counter({'10': 10})


本来は(0,0)を見たいところですが、01と10に制約を課しているため、XYミキサーは01と10の答えのうち正解に近いものを見つけようとします。